# Fetching Census Survey Data

morpc-census connects to the US Census Bureau API, retrieves survey data, and returns it as a normalized long-format DataFrame ready for analysis. Every request is defined by:

- an **Endpoint** — survey dataset and vintage year
- a **scope** — geographic extent (a region, county, state, etc.)
- a **Group** or a list of **variables** — what to retrieve
- an optional **sumlevel** — resolution within the scope (county, tract, place, etc.)

This notebook walks through a typical workflow from discovery to analysis to saving output.

In [1]:
from morpc.logs import config_logs

config_logs('morpc-census-demo.log', 'debug')

2026-05-12 10:58:46,838 | INFO | morpc.logs.config_logs: Set up logging save to file "morpc-census-demo.log", log level "debug"


In [2]:
from morpc_census import (
    Endpoint, Group, CensusAPI, DimensionTable,
    get_all_avail_endpoints,
    IMPLEMENTED_ENDPOINTS, SCOPES,
)
import pandas as pd

## 1. Endpoints — choosing a survey and year

`IMPLEMENTED_ENDPOINTS` lists every survey/table the package supports. `Endpoint` validates the survey name and vintage year at construction time, so you find out immediately if you've specified something unavailable.

In [3]:
# Surveys the package supports
IMPLEMENTED_ENDPOINTS

['acs/acs1',
 'acs/acs1/profile',
 'acs/acs1/subject',
 'acs/acs5',
 'acs/acs5/profile',
 'acs/acs5/subject',
 'dec/pl',
 'dec/dhc',
 'dec/ddhca',
 'dec/ddhcb',
 'dec/sf1',
 'dec/sf2',
 'dec/sf3',
 'geoinfo']

In [4]:
# Construct an endpoint — raises ValueError for unknown surveys or unavailable years
ep = Endpoint('acs/acs5', 2023)
ep

2026-05-12 10:58:51,990 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:58:51,996 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:58:52,210 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 302 None
2026-05-12 10:58:52,281 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 None
2026-05-12 10:58:54,968 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.


Endpoint('acs/acs5', 2023)

In [5]:
# All vintage years available for this survey
ep.vintages

[2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

## 2. Groups — browsing available variables

> **Network required** — the cells below make live calls to the Census API.

A *group* is a table within a survey — a collection of related variables. `endpoint.groups` returns every group with its description. Fetched once, then cached on the object.

In [6]:
# Browse available groups — descriptions keyed by group code
{k: v['description'] for k, v in list(ep.groups.items())[:10]}

2026-05-12 10:59:02,357 | DEBUG | morpc_census.api.groups: Fetching groups for 2023 acs/acs5
2026-05-12 10:59:02,359 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/acs/acs5/groups.json with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:59:02,361 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:59:02,589 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/acs/acs5/groups.json?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 None
2026-05-12 10:59:02,973 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.


{'B01001': 'Sex by Age',
 'B01001A': 'Sex by Age (White Alone)',
 'B01001B': 'Sex by Age (Black or African American Alone)',
 'B01001C': 'Sex by Age (American Indian and Alaska Native Alone)',
 'B01001D': 'Sex by Age (Asian Alone)',
 'B01001E': 'Sex by Age (Native Hawaiian and Other Pacific Islander Alone)',
 'B01001F': 'Sex by Age (Some Other Race Alone)',
 'B01001G': 'Sex by Age (Two or More Races)',
 'B01001H': 'Sex by Age (White Alone, Not Hispanic or Latino)',
 'B01001I': 'Sex by Age (Hispanic or Latino)'}

In [7]:
# Construct a Group — validates the code against endpoint.groups
group = Group(ep, 'B01001')
print(group.description)
print(group.universe)

Sex by Age
Total population


In [8]:
# Variable codes and labels for this group
{k: v.get('label', '') for k, v in list(group.variables.items())[:8]}

2026-05-12 10:59:05,520 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/acs/acs5/groups/B01001.json with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:59:05,523 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:59:05,738 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/acs/acs5/groups/B01001.json?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 19404
2026-05-12 10:59:05,771 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.


{'B01001_001E': 'Estimate!!Total:',
 'B01001_001EA': 'Annotation of Estimate!!Total:',
 'B01001_001M': 'Margin of Error!!Total:',
 'B01001_001MA': 'Annotation of Margin of Error!!Total:',
 'B01001_002E': 'Estimate!!Total:!!Male:',
 'B01001_002EA': 'Annotation of Estimate!!Total:!!Male:',
 'B01001_002M': 'Margin of Error!!Total:!!Male:',
 'B01001_002MA': 'Annotation of Margin of Error!!Total:!!Male:'}

## 3. Scopes — choosing a geographic extent

A scope names the geographic coverage of the request — which places to include. `SCOPES` lists all built-in scopes. Pass a scope key to `CensusAPI` as a string.

In [9]:
# Available scope keys
list(SCOPES.keys())

['us',
 'columbuscbsa',
 'alabama',
 'alaska',
 'arizona',
 'arkansas',
 'california',
 'colorado',
 'connecticut',
 'delaware',
 'district of columbia',
 'florida',
 'georgia',
 'hawaii',
 'idaho',
 'illinois',
 'indiana',
 'iowa',
 'kansas',
 'kentucky',
 'louisiana',
 'maine',
 'maryland',
 'massachusetts',
 'michigan',
 'minnesota',
 'mississippi',
 'missouri',
 'montana',
 'nebraska',
 'nevada',
 'new hampshire',
 'new jersey',
 'new mexico',
 'new york',
 'north carolina',
 'north dakota',
 'ohio',
 'oklahoma',
 'oregon',
 'pennsylvania',
 'rhode island',
 'south carolina',
 'south dakota',
 'tennessee',
 'texas',
 'utah',
 'vermont',
 'virginia',
 'washington',
 'west virginia',
 'wisconsin',
 'wyoming',
 'puerto rico',
 'lake',
 'hancock',
 'allen',
 'morgan',
 'portage',
 'butler',
 'fayette',
 'vinton',
 'paulding',
 'scioto',
 'fulton',
 'henry',
 'logan',
 'brown',
 'monroe',
 'trumbull',
 'pike',
 'pickaway',
 'muskingum',
 'lawrence',
 'adams',
 'crawford',
 'guernsey',
 

## 4. Fetching data

`CensusAPI` validates parameters, builds the request, fetches the data, and transforms it into a normalized long-format table in one step. `api.data` holds the raw wide response from the API; `api.long` is the normalized output.

In [10]:
# Sex by age for all counties in the 15-county MORPC region
b01001 = CensusAPI(ep, 'region15', group='b01001')
b01001.data

2026-05-12 10:59:12,855 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.__init__: Initializing CensusAPI for census-acs-acs5-2023-region15-b01001.
2026-05-12 10:59:12,856 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.__init__: Building request URL and parameters.
2026-05-12 10:59:12,858 | DEBUG | morpc_census.geos.geoids_from_scope: Fetching geoids from scope 'region15'.
2026-05-12 10:59:12,860 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/geoinfo?get=GEO_ID with parameters {'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141', 'in': 'state:39', 'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:59:12,862 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:59:13,311 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/geoinfo?get=GEO_ID&for=county%3A041%2C045%

,B01001_001E,B01001_001EA,B01001_001M,B01001_001MA,B01001_002E,B01001_002EA,B01001_002M,B01001_002MA,B01001_003E,B01001_003EA,...,B01001_048M,B01001_048MA,B01001_049E,B01001_049EA,B01001_049M,B01001_049MA,GEO_ID,NAME,state,county
0,221160,NaN,-555555555,*****,110668,NaN,33,NaN,6456,NaN,...,290,NaN,2009,NaN,352,NaN,0500000US39041,"Delaware County, Ohio",39,41
1,161289,NaN,-555555555,*****,80584,NaN,128,NaN,4690,NaN,...,241,NaN,1892,NaN,225,NaN,0500000US39045,"Fairfield County, Ohio",39,45
2,28880,NaN,-555555555,*****,14271,NaN,163,NaN,778,NaN,...,106,NaN,352,NaN,91,NaN,0500000US39047,"Fayette County, Ohio",39,47
3,1321635,NaN,-555555555,*****,649003,NaN,61,NaN,44950,NaN,...,692,NaN,11174,NaN,747,NaN,0500000US39049,"Franklin County, Ohio",39,49
4,27938,NaN,-555555555,*****,14022,NaN,110,NaN,808,NaN,...,102,NaN,248,NaN,87,NaN,0500000US39073,"Hocking County, Ohio",39,73
5,62888,NaN,-555555555,*****,31021,NaN,150,NaN,1943,NaN,...,175,NaN,895,NaN,160,NaN,0500000US39083,"Knox County, Ohio",39,83
6,180311,NaN,-555555555,*****,89196,NaN,116,NaN,5355,NaN,...,266,NaN,2124,NaN,256,NaN,0500000US39089,"Licking County, Ohio",39,89
7,46140,NaN,-555555555,*****,23206,NaN,100,NaN,1435,NaN,...,158,NaN,464,NaN,107,NaN,0500000US39091,"Logan County, Ohio",39,91
8,44126,NaN,-555555555,*****,24262,NaN,160,NaN,1181,NaN,...,126,NaN,525,NaN,130,NaN,0500000US39097,"Madison County, Ohio",39,97
9,65145,NaN,-555555555,*****,35012,NaN,168,NaN,1943,NaN,...,185,NaN,772,NaN,158,NaN,0500000US39101,"Marion County, Ohio",39,101


## 5. The long-format result

`api.long` is the normalized output: one row per geography × variable. Type suffixes are split into separate columns (`estimate`, `moe`), variable labels are extracted from the raw label strings, and Census missing-value codes are converted to `NaN`.

In [11]:
# Long-format output — one row per geography × variable
b01001.long

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:,B01001_001,221160,-555555555
25,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:,B01001_002,110668,33
48,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!Under 5 years,B01001_003,6456,6
37,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!5 to 9 years,B01001_004,7560,466
26,0500000US39041,"Delaware County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Male:!!10 to 14 years,B01001_005,9200,465
...,...,...,...,...,...,...,...,...,...,...
705,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!67 to 69 years,B01001_045,914,183
706,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!70 to 74 years,B01001_046,1201,211
707,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!75 to 79 years,B01001_047,738,161
708,0500000US39159,"Union County, Ohio",2023,acs/acs5,Sex by age,Total population,Total:!!Female:!!80 to 84 years,B01001_048,571,129


In [12]:
# Column types in the long-format output
b01001.long.dtypes

geoidfq             object
name                object
reference_period     int64
survey              object
concept             object
universe            object
variable_label      object
variable            object
estimate             int64
moe                  int64
dtype: object

## 6. GEOIDFQs — parsing geography identifiers

The `GEOIDFQ` column holds fully-qualified geography identifiers (e.g. `0500000US39049`). `api.geoidfqs` parses each one into a `GeoIDFQ` object with typed fields for summary level, variant, and component codes.

In [13]:
# First three geography IDs parsed as GeoIDFQ objects
b01001.geoidfqs[:3]

[GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='041'),
 GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='045'),
 GeoIDFQ(sumlevel='050', variant='00', geocomp='00', state='39', county='047')]

In [14]:
# Fields on a GeoIDFQ object
g = b01001.geoidfqs[0]
print('summary level:', g.sumlevel)
print('geoid:        ', g.geoid)
print('parts:        ', g.parts)
print('string form:  ', str(g))

summary level: '050'
geoid:         39041
parts:         {'state': '39', 'county': '041'}
string form:   0500000US39041


## 7. Drilling down with sumlevel

The `sumlevel` parameter fetches child geographies within the scope. For example, `sumlevel='tract'` with `scope='franklin'` returns all census tracts in Franklin County.

In [15]:
# Place of birth for all tracts in Franklin County
b05006_tracts = CensusAPI(ep, 'franklin', group='B05006', sumlevel='tract')
b05006_tracts.long

2026-05-12 10:59:26,158 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-countytract-franklin-b05006.__init__: Initializing CensusAPI for census-acs-acs5-2023-countytract-franklin-b05006.
2026-05-12 10:59:26,160 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-countytract-franklin-b05006.__init__: Building request URL and parameters.
2026-05-12 10:59:26,161 | DEBUG | morpc_census.geos.geoids_from_scope: Fetching geoids from scope 'franklin'.
2026-05-12 10:59:26,163 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/geoinfo?get=GEO_ID with parameters {'for': 'county:049', 'in': 'state:39', 'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:59:26,166 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:59:26,623 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/geoinfo?get=GEO_ID&for=county%3A049&in=state%3A39&key=83269ff2

,geoidfq,name,reference_period,survey,concept,universe,variable_label,variable,estimate,moe
0,1400000US39049000110,Census Tract 1.10; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:,B05006_001,60,37
125,1400000US39049000110,Census Tract 1.10; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:,B05006_002,17,19
145,1400000US39049000110,Census Tract 1.10; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:,B05006_003,12,18
146,1400000US39049000110,Census Tract 1.10; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Denmark,B05006_004,0,13
147,1400000US39049000110,Census Tract 1.10; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Europe:!!Northern Europe:!!Ireland,B05006_005,0,13
...,...,...,...,...,...,...,...,...,...,...
58279,1400000US39049980000,Census Tract 9800; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_174,0,13
58276,1400000US39049980000,Census Tract 9800; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Latin America:!!South Ameri...,B05006_175,0,13
58280,1400000US39049980000,Census Tract 9800; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:,B05006_176,0,13
58281,1400000US39049980000,Census Tract 9800; Franklin County; Ohio,2023,acs/acs5,Place of birth for the foreign-born population...,Foreign-born population excluding population b...,Total:!!Americas:!!Northern America:!!Canada,B05006_177,0,13


## 8. Fetching specific variables without a group

When you only need a few variables — possibly from different groups — pass `variables` directly instead of a group. The Census API allows at most 50 fields per request; the package batches automatically when you exceed that limit.

In [16]:
# Total population, male, and female — fetched directly without a group
pop = CensusAPI(ep, 'region15', variables=['B01001_001E', 'B01001_002E', 'B01001_026E'])
pop.long

2026-05-12 10:59:33,273 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-select-variables.__init__: Initializing CensusAPI for census-acs-acs5-2023-region15-select-variables.
2026-05-12 10:59:33,274 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-select-variables.__init__: Building request URL and parameters.
2026-05-12 10:59:33,275 | DEBUG | morpc_census.geos.geoids_from_scope: Fetching geoids from scope 'region15'.
2026-05-12 10:59:33,279 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2023/geoinfo?get=GEO_ID with parameters {'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141', 'in': 'state:39', 'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 10:59:33,282 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 10:59:33,715 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2023/geoinfo?get=

,geoidfq,reference_period,survey,concept,universe,variable_label,variable,estimate
0,0500000US39041,2023,acs/acs5,,Not defined — no group specified,Total:,B01001_001,221160
2,0500000US39041,2023,acs/acs5,,Not defined — no group specified,Total:!!Male:,B01001_002,110668
1,0500000US39041,2023,acs/acs5,,Not defined — no group specified,Total:!!Female:,B01001_026,110492
3,0500000US39045,2023,acs/acs5,,Not defined — no group specified,Total:,B01001_001,161289
5,0500000US39045,2023,acs/acs5,,Not defined — no group specified,Total:!!Male:,B01001_002,80584
4,0500000US39045,2023,acs/acs5,,Not defined — no group specified,Total:!!Female:,B01001_026,80705
6,0500000US39047,2023,acs/acs5,,Not defined — no group specified,Total:,B01001_001,28880
8,0500000US39047,2023,acs/acs5,,Not defined — no group specified,Total:!!Male:,B01001_002,14271
7,0500000US39047,2023,acs/acs5,,Not defined — no group specified,Total:!!Female:,B01001_026,14609
9,0500000US39049,2023,acs/acs5,,Not defined — no group specified,Total:,B01001_001,1321635


## 9. Analyzing with DimensionTable

`DimensionTable` parses each variable's `!!`-delimited label into named **dimension columns** and pivots the long-format data into a readable wide table.

Each label segment is classified as a *subtotal* (ends with `:`) or a *leaf* (no trailing `:`). Subtotals are left-aligned into the first columns; leaves into the last. This alignment keeps the same concept in the same column regardless of path depth — for example, in B05004 (Nativity and Citizenship × Sex), *Male* and *Female* always land in the last column whether there are one or two nativity levels before them.

For B01001 (Sex by Age), `Total:!!Male:!!Under 5 years` parses as:

| dim | value |
|-----|-------|
| `total` | `Total:` |
| `sex` | `Male:` |
| `age` | `Under 5 years` |

Key methods:
- `drop(dim, method='summarize')` — remove a dimension level before pivoting
- `remap(variable_map)` — relabel and collapse variables, then re-parse dimensions
- `wide()` — produce the wide pivot table
- `percent()` — column percentages relative to the grand total row

In [17]:
# Construct a DimensionTable; dim_names labels the parsed dimension columns
dim = DimensionTable(b01001.long, dim_names=['total', 'sex', 'age'])

# Each row is one Census variable; each column is a level of the label hierarchy
# ':' suffix is preserved here — it marks subtotals (rows that have sub-dimensions)
dim.dims.head(8)

,total,sex,age
variable,,,
B01001_001,Total:,,
B01001_002,Total:,Male:,
B01001_003,Total:,Male:,Under 5 years
B01001_004,Total:,Male:,5 to 9 years
B01001_005,Total:,Male:,10 to 14 years
B01001_006,Total:,Male:,15 to 17 years
B01001_007,Total:,Male:,18 and 19 years
B01001_008,Total:,Male:,20 years


In [18]:
# Full wide table — all dimension levels as row index, geography × value type as columns
# ':' is stripped from displayed index values (Total: → Total, Male: → Male, etc.)
dim.wide()

estimate                   moe  \
geoidfq                               0500000US39041        0500000US39041   
name                           Delaware County, Ohio Delaware County, Ohio   
reference_period                                2023                  2023   
survey                                      acs/acs5              acs/acs5   
concept                                   Sex by age            Sex by age   
universe                            Total population      Total population   
total sex    age                                                             
Total                                         221160            -555555555   
      Male                                    110668                    33   
             Under 5 years                      6456                     6   
             5 to 9 years                       7560                   466   
             10 to 14 years                     9200                   465   
             15 to 17 years                     5478                    22   
             18 and 19 years                    3068                   176   
             20 years                           1337                   298   
             21 years                           1490                   305   
             22 to 24 years                     3434                   362   
             25 to 29 years                     4556                   140   
             30 to 34 years                     5846                    20   
             35 to 39 years                     7477                   606   
             40 to 44 years                     9403                   609   
             45 to 49 years                     8681                    19   
             50 to 54 years                     8309                    26   
             55 to 59 years                     7351                   547   
             60 and 61 years                    2678                   457   
             62 to 64 years                     3255                   382   
             65 and 66 years                    2304                   346   
             67 to 69 years                     2872                   310   
             70 to 74 years                     4734                   364   
             75 to 79 years                     2642                   284   
             80 to 84 years                     1212                   179   
             85 years and over                  1325                   273   
      Female                                  110492                    33   
             Under 5 years                      6078                    23   
             5 to 9 years                       7775                   413   
             10 to 14 years                     8150                   411   
             15 to 17 years                     5177                    19   
             18 and 19 years                    2782                    97   
             20 years                           1353                   306   
             21 years                           1370                   325   
             22 to 24 years                     3133                   410   
             25 to 29 years                     4701                   136   
             30 to 34 years                     6186                    85   
             35 to 39 years                     8321                   550   
             40 to 44 years                     8575                   547   
             45 to 49 years                     8251                    15   
             50 to 54 years                     7863                    15   
             55 to 59 years                     7343                   397   
             60 and 61 years                    2459                   309   
             62 to 64 years                     3557                   393   
             65 and 66 years                    2722              

In [19]:
# Drop the age dimension: keep only the pre-computed sex-level rows
# (grand total + Male total + Female total — one row per sex per county)
dim.drop('age').wide()

estimate                   moe  \
geoidfq                 0500000US39041        0500000US39041   
name             Delaware County, Ohio Delaware County, Ohio   
reference_period                  2023                  2023   
survey                        acs/acs5              acs/acs5   
concept                     Sex by age            Sex by age   
universe              Total population      Total population   
total sex                                                      
Total                           221160            -555555555   
      Male                      110668                    33   
      Female                    110492                    33   

                               estimate                    moe  \
geoidfq                  0500000US39045         0500000US39045   
name             Fairfield County, Ohio Fairfield County, Ohio   
reference_period                   2023                   2023   
survey                         acs/acs5               acs/acs5   
concept                      Sex by age             Sex by age   
universe               Total population       Total population   
total sex                                                        
Total                            161289             -555555555   
      Male                        80584                    128   
      Female                      80705                    128   

                             estimate                  moe  \
geoidfq                0500000US39047       0500000US39047   
name             Fayette County, Ohio Fayette County, Ohio   
reference_period                 2023                 2023   
survey                       acs/acs5             acs/acs5   
concept                    Sex by age           Sex by age   
universe             Total population     Total population   
total sex                                                    
Total                           28880           -555555555   
      Male                      14271                  163   
      Female                    14609                  163   

                              estimate                   moe  \
geoidfq                 0500000US39049        0500000US39049   
name             Franklin County, Ohio Franklin County, Ohio   
reference_period                  2023                  2023   
survey                        acs/acs5              acs/acs5   
concept                     Sex by age            Sex by age   
universe              Total population      Total population   
total sex                                                      
Total                          1321635            -555555555   
      Male                      649003                    61   
      Female                    672632                    61   

                             estimate                  moe  ...  \
geoidfq                0500000US39073       0500000US39073  ...   
name             Hocking County, Ohio Hocking County, Ohio  ...   
reference_period                 2023                 2023  ...   
survey                       acs/acs5             acs/acs5  ...   
concept                    Sex by age           Sex by age  ...   
universe             Total population     Total population  ...   
total sex                                                   ...   
Total                           27938           -555555555  ...   
      Male                      14022                  110  ...   
      Female                    13916                  110  ...   

                            estimate                 moe           estimate  \
geoidfq               0500000US39117      0500000US39117     0500000US39127   
name             Morrow County, Ohio Morrow County, Ohio Perry County, Ohio   
reference_period                2023                2023               2023   
survey                      acs/acs5            acs/acs5           acs/acs5   
concept                   Sex by age          Sex by age         Sex by age  

In [20]:
from morpc_census import AGEGROUP_MAP

# B01001 uses uneven age bins: "15 to 17 years" and "18 and 19 years" are separate
# variables, as are "20 years", "21 years", and "22 to 24 years".
# AGEGROUP_MAP collapses these into standard 5-year groups. remap() applies the
# substitutions and sums estimates for any rows that share a new label.
dim_remapped = DimensionTable(b01001.long, dim_names=['total', 'sex', 'age'])
dim_remapped.remap(AGEGROUP_MAP)

# Population by standardized age group — aggregate sex out (sum Male + Female)
dim_remapped.drop('sex', method='aggregate').wide()

2026-05-12 11:00:06,920 | INFO | morpc_census.api.DimensionTable.remap: Remapping variables: ['Under 5 years', '5 to 9 years', '10 to 14 years', '15 to 17 years', '18 and 19 years', '20 years', '21 years', '22 to 24 years', '25 to 29 years', '30 to 34 years', '35 to 39 years', '40 to 44 years', '45 to 49 years', '50 to 54 years', '55 to 59 years', '60 and 61 years', '62 to 64 years', '65 and 66 years', '67 to 69 years', '70 to 74 years', '75 to 79 years', '80 to 84 years', '85 years and over'].


estimate                   moe  \
geoidfq                        0500000US39041        0500000US39041   
name                    Delaware County, Ohio Delaware County, Ohio   
reference_period                         2023                  2023   
survey                               acs/acs5              acs/acs5   
concept                            Sex by age            Sex by age   
universe                     Total population      Total population   
total age                                                             
Total                                221160.0             46.669048   
      10 to 14 years                  17350.0            620.601321   
      15 to 19 years                  16505.0            203.051718   
      20 to 24 years                  12117.0            824.762996   
      25 to 29 years                   9257.0            195.181966   
      30 to 34 years                  12032.0             87.321246   
      35 to 39 years                  15798.0            818.373998   
      40 to 44 years                  17978.0            818.590252   
      45 to 49 years                  16932.0             24.207437   
      5 to 9 years                    15335.0            622.675678   
      50 to 54 years                  16172.0             30.016662   
      55 to 59 years                  14694.0            675.883126   
      60 to 64 years                  11949.0            777.626517   
      65 to 69 years                  10835.0            647.898912   
      70 to 74 years                   9472.0            522.609797   
      75 to 79 years                   6050.0            448.402721   
      80 to 84 years                   2816.0            340.794660   
      85 years and over                3334.0            445.458191   
      Under 5 years                   12534.0             23.769729   

                                      estimate                    moe  \
geoidfq                         0500000US39045         0500000US39045   
name                    Fairfield County, Ohio Fairfield County, Ohio   
reference_period                          2023                   2023   
survey                                acs/acs5               acs/acs5   
concept                             Sex by age             Sex by age   
universe                      Total population       Total population   
total age                                                               
Total                                 161289.0             181.019336   
      10 to 14 years                   11973.0             651.871153   
      15 to 19 years                   10650.0             231.877123   
      20 to 24 years                    9488.0             608.583601   
      25 to 29 years                    9358.0             150.000000   
      30 to 34 years                    9928.0             116.846053   
      35 to 39 years                   11009.0             603.334899   
      40 to 44 years                   10701.0             640.800281   
      45 to 49 years                   10357.0             158.492902   
      5 to 9 years                     10230.0             651.871153   
      50 to 54 years                   10787.0             121.938509   
      55 to 59 years                   10925.0             589.512510   
      60 to 64 years                   10135.0             687.373261   
      65 to 69 years                    8791.0             541.505309   
      70 to 74 years                    6820.0             410.128029   
      75 to 79 years                    4476.0             333.594065   
      80 to 84 years                    3340.0             297.249054   
      85 years and over                 3020.0             311.899022   
      Under 5 years                     9301.0             121.622366   

                                    estimate                  moe  \
geoidfq                       0500000US39047       0500000US39047   
name     

In [21]:
# Sex percentages relative to the county total (age dropped; grand total = 100%)
dim.drop('age').percent()

estimate                   moe  \
geoidfq                 0500000US39041        0500000US39041   
name             Delaware County, Ohio Delaware County, Ohio   
reference_period                  2023                  2023   
survey                        acs/acs5              acs/acs5   
concept                     Sex by age            Sex by age   
universe              Total population      Total population   
total sex                                                      
Total Male                       50.04                  -0.0   
      Female                     49.96                  -0.0   

                               estimate                    moe  \
geoidfq                  0500000US39045         0500000US39045   
name             Fairfield County, Ohio Fairfield County, Ohio   
reference_period                   2023                   2023   
survey                         acs/acs5               acs/acs5   
concept                      Sex by age             Sex by age   
universe               Total population       Total population   
total sex                                                        
Total Male                        49.96                   -0.0   
      Female                      50.04                   -0.0   

                             estimate                  moe  \
geoidfq                0500000US39047       0500000US39047   
name             Fayette County, Ohio Fayette County, Ohio   
reference_period                 2023                 2023   
survey                       acs/acs5             acs/acs5   
concept                    Sex by age           Sex by age   
universe             Total population     Total population   
total sex                                                    
Total Male                      49.41                 -0.0   
      Female                    50.59                 -0.0   

                              estimate                   moe  \
geoidfq                 0500000US39049        0500000US39049   
name             Franklin County, Ohio Franklin County, Ohio   
reference_period                  2023                  2023   
survey                        acs/acs5              acs/acs5   
concept                     Sex by age            Sex by age   
universe              Total population      Total population   
total sex                                                      
Total Male                       49.11                  -0.0   
      Female                     50.89                  -0.0   

                             estimate                  moe  ...  \
geoidfq                0500000US39073       0500000US39073  ...   
name             Hocking County, Ohio Hocking County, Ohio  ...   
reference_period                 2023                 2023  ...   
survey                       acs/acs5             acs/acs5  ...   
concept                    Sex by age           Sex by age  ...   
universe             Total population     Total population  ...   
total sex                                                   ...   
Total Male                      50.19                 -0.0  ...   
      Female                    49.81                 -0.0  ...   

                            estimate                 moe           estimate  \
geoidfq               0500000US39117      0500000US39117     0500000US39127   
name             Morrow County, Ohio Morrow County, Ohio Perry County, Ohio   
reference_period                2023                2023               2023   
survey                      acs/acs5            acs/acs5           acs/acs5   
concept                   Sex by age          Sex by age         Sex by age   
universe            Total population    Total population   Total population   
total sex                                                                     
Total Male                     50.53                -0.0              50.12   
      Female                   49.47                -0.0              49.88   

    

## 10. Time series

`DimensionTable` accepts any long-format DataFrame with the standard schema. Concatenating `long` from multiple vintages produces a time-series table where `reference_period` becomes a column dimension.

In [22]:
# Fetch the same group for an earlier vintage
ep2018 = Endpoint('acs/acs5', 2018)
b01001_2018 = CensusAPI(ep2018, 'region15', group=Group(ep2018, 'B01001'))

# Stack the two years and build a dimension table
long_ts = pd.concat([b01001_2018.long, b01001.long])
DimensionTable(long_ts).wide()

2026-05-12 11:00:09,909 | DEBUG | morpc_census.api.groups: Fetching groups for 2018 acs/acs5
2026-05-12 11:00:09,912 | DEBUG | morpc.req.get_json_safely: Getting data from https://api.census.gov/data/2018/acs/acs5/groups.json with parameters {'key': '83269ff2739cb3bd485c75b091dcee493ad6fe70'}.
2026-05-12 11:00:09,915 | DEBUG | urllib3.connectionpool._new_conn: Starting new HTTPS connection (1): api.census.gov:443
2026-05-12 11:00:10,117 | DEBUG | urllib3.connectionpool._make_request: https://api.census.gov:443 "GET /data/2018/acs/acs5/groups.json?key=83269ff2739cb3bd485c75b091dcee493ad6fe70 HTTP/1.1" 200 None
2026-05-12 11:00:10,478 | DEBUG | morpc.req.get_json_safely: Request successful. Decoding return JSON.
2026-05-12 11:00:10,486 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2018-region15-b01001.__init__: Initializing CensusAPI for census-acs-acs5-2018-region15-b01001.
2026-05-12 11:00:10,488 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2018-region15-b01001.__init__: B

estimate                   \
geoidfq                               0500000US39041                    
name                           Delaware County, Ohio                    
reference_period                                2018             2023   
survey                                      acs/acs5         acs/acs5   
concept                                   Sex by age       Sex by age   
universe                            Total population Total population   
dim_0 dim_1  dim_2                                                      
Total                                         197008           221160   
      Male                                     97504           110668   
             Under 5 years                      6273             6456   
             5 to 9 years                       7752             7560   
             10 to 14 years                     8435             9200   
             15 to 17 years                     4731             5478   
             18 and 19 years                    2531             3068   
             20 years                           1515             1337   
             21 years                           1000             1490   
             22 to 24 years                     2824             3434   
             25 to 29 years                     3911             4556   
             30 to 34 years                     5349             5846   
             35 to 39 years                     7256             7477   
             40 to 44 years                     7690             9403   
             45 to 49 years                     8006             8681   
             50 to 54 years                     7175             8309   
             55 to 59 years                     6484             7351   
             60 and 61 years                    2254             2678   
             62 to 64 years                     2915             3255   
             65 and 66 years                    1892             2304   
             67 to 69 years                     2676             2872   
             70 to 74 years                     2946             4734   
             75 to 79 years                     1655             2642   
             80 to 84 years                     1296             1212   
             85 years and over                   938             1325   
      Female                                   99504           110492   
             Under 5 years                      6112             6078   
             5 to 9 years                       7693             7775   
             10 to 14 years                     7624             8150   
             15 to 17 years                     4612             5177   
             18 and 19 years                    2375             2782   
             20 years                           1086             1353   
             21 years                            973             1370   
             22 to 24 years                     2928             3133   
             25 to 29 years                     4177             4701   
             30 to 34 years                     6046             6186   
             35 to 39 years                     7589             8321   
             40 to 44 years                     7861             8575   
             45 to 49 years                     7741             8251   
             50 to 54 years                     7104             7863   
             55 to 59 years                     6412             7343   
             60 and 61 years                    2287             2459   
             62 to 64 years                     3262             3557   
             65 and 66 years                    1653             2722   
             67 to 69 years                     2828             2937   
             70 to 74 years                     3630             4738   
             75 to 79 years                     2458             3408   
             80 to 84 years  

## 11. Saving output

`api.save()` writes three files to the output directory:

- `{name}.long.csv` — the long-format data
- `{name}.schema.yaml` — a frictionless Schema describing every column
- `{name}.resource.yaml` — a frictionless Resource descriptor linking the CSV to its schema

The resource is validated immediately after writing, so any schema mismatch surfaces here.

In [23]:
import os

b01001.save('./temp_data')

print('filename:', b01001.filename)
print('exists:  ', os.path.exists(f'./temp_data/{b01001.filename}'))

2026-05-12 11:00:17,318 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.save: Writing data to temp_data/census-acs-acs5-2023-region15-b01001.long.csv.
2026-05-12 11:00:17,326 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.save: Writing schema to temp_data/census-acs-acs5-2023-region15-b01001.schema.yaml.
2026-05-12 11:00:17,327 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.define_schema: Defining schema for acs/acs5 / 2023 / B01001.
2026-05-12 11:00:18,572 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.save: Writing resource to temp_data/census-acs-acs5-2023-region15-b01001.resource.yaml.
2026-05-12 11:00:18,613 | INFO | morpc_census.api.CensusAPI.census-acs-acs5-2023-region15-b01001.save: Save complete and resource validated.
filename: census-acs-acs5-2023-region15-b01001.long.csv
exists:   True


In [24]:
from frictionless import Resource

Resource(f'temp_data/{b01001.name}.resource.yaml')

{'name': 'census-acs-acs5-2023-region15-b01001',
 'type': 'table',
 'title': '2023 Sex by Age for region15',
 'description': 'Census API data for B01001: Sex by Age from acs/acs5 2023 for '
                'region15.',
 'sources': [{'title': 'US Census Bureau API',
              'path': 'https://api.census.gov/data/2023/acs/acs5?',
              '_params': {'get': 'group(B01001)',
                          'for': 'county:041,045,049,089,097,129,159,083,101,117,047,073,091,127,141',
                          'in': 'state:39'}}],
 'path': 'census-acs-acs5-2023-region15-b01001.long.csv',
 'scheme': 'file',
 'format': 'csv',
 'mediatype': 'text/csv',
 'schema': 'census-acs-acs5-2023-region15-b01001.schema.yaml'}

In [25]:
from frictionless import Schema

Schema(f'temp_data/{b01001.name}.schema.yaml')

{'fields': [{'name': 'geoidfq',
             'type': 'string',
             'description': 'Census geography fully-qualified identifier'},
            {'name': 'name', 'type': 'string', 'description': 'Geography name'},
            {'name': 'reference_period',
             'type': 'integer',
             'description': 'Reference year'},
            {'name': 'survey',
             'type': 'string',
             'description': 'Census survey endpoint'},
            {'name': 'concept',
             'type': 'string',
             'description': 'Table concept description'},
            {'name': 'universe',
             'type': 'string',
             'description': 'Universe for the table'},
            {'name': 'variable_label',
             'type': 'string',
             'description': 'Human-readable variable label'},
            {'name': 'variable',
             'type': 'string',
             'description': 'Base variable code'},
            {'name': 'estimate',
             'type': 'n